# Fakeddit dataset - EDA

In [ ]:
!pip install -q kaggle

In [ ]:
!mkdir -p /kaggle/data/fakeddit
!kaggle datasets download -d vanshikavmittal/fakeddit-dataset -p /kaggle/data/fakeddit --unzip

In [ ]:
import polars as pl
from pathlib import Path
import plotly.express as px
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib

In [ ]:
SAMPLES_DIR = Path("/kaggle/data/fakeddit/multimodal_only_samples")

all_data: pl.DataFrame = pl.concat([
    pl.read_csv(SAMPLES_DIR / filename, separator="\t") for filename in SAMPLES_DIR.iterdir()
]).with_columns(
    pl.from_epoch("created_utc", time_unit="s").alias("created_date")
)

In [ ]:
import os

cpu_count = os.cpu_count()
print(cpu_count)

In [ ]:
import asyncio
import random
import tarfile
import time
from collections import Counter
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from io import BytesIO
from pathlib import Path
from typing import Literal, TypedDict, Union
from urllib.parse import urlparse

import aiohttp
import polars as pl
from PIL import Image
from tqdm.auto import tqdm


# ============================================================
# Conservative download settings
# ============================================================

MAX_IN_FLIGHT = 256
PER_HOST_LIMIT = 60

GLOBAL_REQUESTS_PER_SECOND = 200
PER_HOST_REQUESTS_PER_SECOND = 60

REQUEST_JITTER_SECONDS = 0.05

WRITE_WORKERS = 16
WRITE_QUEUE_SIZE = WRITE_WORKERS * 8

CONNECT_TIMEOUT = 2
READ_TIMEOUT = 2

MAX_RETRIES = 3
BASE_BACKOFF_SECONDS = 0.75
MAX_BACKOFF_SECONDS = 8.0
BACKOFF_FACTOR = 2.0

RETRY_HTTP_STATUSES = {408, 429, 500, 502, 503, }

SAVE_DIR = Path("/kaggle/images/")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_WORKING_DIR = Path("/kaggle/working")
ARCHIVE_PATH = KAGGLE_WORKING_DIR / "images.tar.xz"


# ============================================================
# User agents for testing
# ============================================================

BASE_USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:151.0) "
    "Gecko/20100101 Firefox/151.0"
)

USER_AGENTS = [
    f"{BASE_USER_AGENT} test-{i}"
    for i in range(1, 100)
]


# ============================================================
# Result types
# ============================================================

class ImageOk(TypedDict):
    status: Literal["ok"]
    image_url: str
    image_path: str
    image_id: int
    user_agent: str


class ImageError(TypedDict):
    status: Literal["error"]
    image_url: str
    exception_type: str
    exception: str
    http_status: int | None
    user_agent: str | None


ImageResult = Union[ImageOk, ImageError]


class FetchOk(TypedDict):
    status: Literal["ok"]
    image_id: int
    image_url: str
    data: bytes
    content_type: str | None
    user_agent: str


FetchResult = Union[FetchOk, ImageError]


PIL_FORMAT_TO_EXT = {
    "JPEG": ".jpg",
    "PNG": ".png",
    "WEBP": ".webp",
    "GIF": ".gif",
    "BMP": ".bmp",
    "AVIF": ".avif",
}


# ============================================================
# Rate limiting
# ============================================================

class AsyncRateLimiter:
    def __init__(self, requests_per_second: float, jitter_seconds: float = 0.0):
        if requests_per_second <= 0:
            raise ValueError("requests_per_second must be > 0")

        self.min_interval = 1.0 / requests_per_second
        self.jitter_seconds = jitter_seconds
        self.lock = asyncio.Lock()
        self.last_request_at = 0.0

    async def wait(self) -> None:
        async with self.lock:
            loop = asyncio.get_running_loop()
            now = loop.time()

            wait_for = self.min_interval - (now - self.last_request_at)
            if wait_for > 0:
                await asyncio.sleep(wait_for)

            if self.jitter_seconds > 0:
                await asyncio.sleep(random.uniform(0, self.jitter_seconds))

            self.last_request_at = loop.time()


class PerHostRateLimiters:
    def __init__(self, requests_per_second: float, jitter_seconds: float = 0.0):
        self.requests_per_second = requests_per_second
        self.jitter_seconds = jitter_seconds
        self.limiters: dict[str, AsyncRateLimiter] = {}
        self.lock = asyncio.Lock()

    async def wait(self, url: str) -> None:
        host = urlparse(url).netloc.lower() or "unknown-host"

        async with self.lock:
            limiter = self.limiters.get(host)

            if limiter is None:
                limiter = AsyncRateLimiter(
                    requests_per_second=self.requests_per_second,
                    jitter_seconds=self.jitter_seconds,
                )
                self.limiters[host] = limiter

        await limiter.wait()


# ============================================================
# Helpers
# ============================================================

def validate_and_save_image_bytes(
    image_id: int,
    data: bytes,
) -> str:
    try:
        with Image.open(BytesIO(data)) as img:
            image_format = img.format
            img.load()

    except Exception as e:
        raise ValueError(f"PIL can't load image: {e}") from e

    ext = PIL_FORMAT_TO_EXT.get(image_format, ".img")
    path = SAVE_DIR / f"{image_id:012d}{ext}"
    path.write_bytes(data)

    return str(path)


def extension_from_url(url: str) -> str:
    suffix = Path(urlparse(url).path).suffix.lower()

    if suffix in {".jpg", ".jpeg", ".png", ".webp", ".gif", ".bmp", ".avif"}:
        return suffix

    return ".img"


def extension_from_content_type(content_type: str) -> str:
    content_type = content_type.split(";", 1)[0].strip().lower()

    return {
        "image/jpeg": ".jpg",
        "image/png": ".png",
        "image/webp": ".webp",
        "image/gif": ".gif",
        "image/bmp": ".bmp",
        "image/avif": ".avif",
    }.get(content_type, ".img")


def image_path_for_id(
    image_id: int,
    url: str,
    content_type: str | None = None,
) -> Path:
    ext = extension_from_url(url)

    if ext == ".img" and content_type:
        ext = extension_from_content_type(content_type)

    return SAVE_DIR / f"{image_id:012d}{ext}"


def exception_text(e: BaseException) -> str:
    return str(e) or repr(e)


def parse_retry_after_seconds(value: str | None) -> float | None:
    if not value:
        return None

    value = value.strip()

    if value.isdigit():
        return float(value)

    try:
        retry_time = parsedate_to_datetime(value)

        if retry_time.tzinfo is None:
            retry_time = retry_time.replace(tzinfo=timezone.utc)

        now = datetime.now(timezone.utc)
        return max(0.0, (retry_time - now).total_seconds())

    except Exception:
        return None


def backoff_seconds(attempt: int, retry_after: str | None = None) -> float:
    retry_after_seconds = parse_retry_after_seconds(retry_after)

    if retry_after_seconds is not None:
        return min(retry_after_seconds, MAX_BACKOFF_SECONDS)

    deterministic = BASE_BACKOFF_SECONDS * (BACKOFF_FACTOR ** attempt)
    jitter = random.uniform(0, REQUEST_JITTER_SECONDS)

    return min(deterministic + jitter, MAX_BACKOFF_SECONDS)


def choose_user_agent() -> str:
    return random.choice(USER_AGENTS)


def make_headers(user_agent: str) -> dict[str, str]:
    return {
        "User-Agent": user_agent,
        "Accept": "image/avif,image/webp,image/apng,image/*,*/*;q=0.8",
        "Connection": "keep-alive",
    }


# ============================================================
# Fetching
# ============================================================

async def fetch_image_bytes(
    session: aiohttp.ClientSession,
    image_id: int,
    url: str,
    global_rate_limiter: AsyncRateLimiter,
    per_host_rate_limiters: PerHostRateLimiters,
) -> FetchResult:
    last_exception_type = "UnknownError"
    last_exception = "Unknown error"
    last_http_status: int | None = None
    last_user_agent: str | None = None

    for attempt in range(MAX_RETRIES + 1):
        user_agent = choose_user_agent()
        last_user_agent = user_agent

        try:
            await global_rate_limiter.wait()
            await per_host_rate_limiters.wait(url)

            async with session.get(
                url,
                headers=make_headers(user_agent),
            ) as response:
                http_status = response.status
                last_http_status = http_status
                content_type = response.headers.get("Content-Type")

                if http_status >= 400:
                    response.release()

                    last_exception_type = "HTTPStatusError"
                    last_exception = f"HTTP status {http_status}"

                    if http_status in RETRY_HTTP_STATUSES and attempt < MAX_RETRIES:
                        sleep_for = backoff_seconds(
                            attempt=attempt,
                            retry_after=response.headers.get("Retry-After"),
                        )
                        await asyncio.sleep(sleep_for)
                        continue

                    return {
                        "status": "error",
                        "image_url": url,
                        "exception_type": last_exception_type,
                        "exception": last_exception,
                        "http_status": http_status,
                        "user_agent": user_agent,
                    }

                data = await response.read()

                return {
                    "status": "ok",
                    "image_id": image_id,
                    "image_url": url,
                    "data": data,
                    "content_type": content_type,
                    "user_agent": user_agent,
                }

        except asyncio.TimeoutError as e:
            last_exception_type = "TimeoutError"
            last_exception = exception_text(e)
            last_http_status = None

            if attempt < MAX_RETRIES:
                await asyncio.sleep(backoff_seconds(attempt))
                continue

        except aiohttp.ClientError as e:
            last_exception_type = type(e).__name__
            last_exception = exception_text(e)
            last_http_status = None

            if attempt < MAX_RETRIES:
                await asyncio.sleep(backoff_seconds(attempt))
                continue

        except Exception as e:
            last_exception_type = type(e).__name__
            last_exception = exception_text(e)
            last_http_status = None
            break

    return {
        "status": "error",
        "image_url": url,
        "exception_type": last_exception_type,
        "exception": last_exception,
        "http_status": last_http_status,
        "user_agent": last_user_agent,
    }


async def fetch_worker(
    url_queue: asyncio.Queue,
    write_queue: asyncio.Queue,
    session: aiohttp.ClientSession,
    global_rate_limiter: AsyncRateLimiter,
    per_host_rate_limiters: PerHostRateLimiters,
) -> None:
    while True:
        item = await url_queue.get()

        if item is None:
            url_queue.task_done()
            return

        image_id, url = item

        result = await fetch_image_bytes(
            session=session,
            image_id=image_id,
            url=url,
            global_rate_limiter=global_rate_limiter,
            per_host_rate_limiters=per_host_rate_limiters,
        )

        await write_queue.put(result)
        url_queue.task_done()


async def write_worker(
    write_queue: asyncio.Queue,
    results: list[ImageResult],
    exception_counts: Counter,
    http_status_counts: Counter,
    user_agent_counts: Counter,
    pbar: tqdm,
) -> None:
    while True:
        result = await write_queue.get()

        if result is None:
            write_queue.task_done()
            return

        user_agent = result.get("user_agent")
        if user_agent:
            user_agent_counts[user_agent] += 1

        if result["status"] == "ok":
            try:
                image_path = await asyncio.to_thread(
                    validate_and_save_image_bytes,
                    result["image_id"],
                    result["data"],
                )

                results.append({
                    "status": "ok",
                    "image_url": result["image_url"],
                    "image_path": image_path,
                    "image_id": result["image_id"],
                    "user_agent": result["user_agent"],
                })

            except Exception as e:
                exception_type = type(e).__name__

                exception_counts[exception_type] += 1
                http_status_counts[None] += 1

                results.append({
                    "status": "error",
                    "image_url": result["image_url"],
                    "exception_type": exception_type,
                    "exception": exception_text(e),
                    "http_status": None,
                    "user_agent": result["user_agent"],
                })

        else:
            exception_counts[result["exception_type"]] += 1
            http_status_counts[result["http_status"]] += 1
            results.append(result)

        pbar.update(1)
        write_queue.task_done()


# ============================================================
# Metadata and archive
# ============================================================

def write_results_metadata(
    results: list[ImageResult],
    exception_counts: Counter,
    http_status_counts: Counter,
    user_agent_counts: Counter,
    elapsed: float,
) -> None:
    results_path = SAVE_DIR / "fetch_results.parquet"
    summary_path = SAVE_DIR / "fetch_summary.txt"

    if results:
        pl.DataFrame(results).write_parquet(results_path)

    ok_count = sum(1 for row in results if row["status"] == "ok")
    error_count = len(results) - ok_count

    summary_text = [
        f"total_results: {len(results)}",
        f"ok: {ok_count}",
        f"errors: {error_count}",
        f"elapsed_seconds: {elapsed:.2f}",
        "",
        "settings:",
        f"  MAX_IN_FLIGHT: {MAX_IN_FLIGHT}",
        f"  PER_HOST_LIMIT: {PER_HOST_LIMIT}",
        f"  GLOBAL_REQUESTS_PER_SECOND: {GLOBAL_REQUESTS_PER_SECOND}",
        f"  PER_HOST_REQUESTS_PER_SECOND: {PER_HOST_REQUESTS_PER_SECOND}",
        f"  REQUEST_JITTER_SECONDS: {REQUEST_JITTER_SECONDS}",
        f"  MAX_RETRIES: {MAX_RETRIES}",
        f"  BASE_BACKOFF_SECONDS: {BASE_BACKOFF_SECONDS}",
        f"  MAX_BACKOFF_SECONDS: {MAX_BACKOFF_SECONDS}",
        "",
        "exception_counts:",
    ]

    for key, value in exception_counts.most_common():
        summary_text.append(f"  {key}: {value}")

    summary_text.append("")
    summary_text.append("http_status_counts:")

    for key, value in http_status_counts.most_common():
        summary_text.append(f"  {key}: {value}")

    summary_text.append("")
    summary_text.append("user_agent_counts:")

    for key, value in user_agent_counts.most_common():
        summary_text.append(f"  {key}: {value}")

    summary_path.write_text("\n".join(summary_text), encoding="utf-8")


def make_tar_xz(source_dir: Path, archive_path: Path) -> Path:
    archive_path.parent.mkdir(parents=True, exist_ok=True)

    if archive_path.exists():
        archive_path.unlink()

    with tarfile.open(archive_path, mode="w:xz") as tar:
        tar.add(source_dir, arcname=source_dir.name)
    print(f"YAY DONE  {archive_path}")

    return archive_path


# ============================================================
# Main API
# ============================================================

async def fetch_all_images(
    urls: list[str],
) -> tuple[list[ImageResult], Counter, Counter, Counter, float, str]:
    results: list[ImageResult] = []
    exception_counts = Counter()
    http_status_counts = Counter()
    user_agent_counts = Counter()

    timeout = aiohttp.ClientTimeout(
        total=None,
        connect=CONNECT_TIMEOUT,
        sock_read=READ_TIMEOUT,
    )

    connector = aiohttp.TCPConnector(
        limit=MAX_IN_FLIGHT,
        limit_per_host=PER_HOST_LIMIT,
        ttl_dns_cache=300,
        enable_cleanup_closed=True,
    )

    global_rate_limiter = AsyncRateLimiter(
        requests_per_second=GLOBAL_REQUESTS_PER_SECOND,
        jitter_seconds=REQUEST_JITTER_SECONDS,
    )

    per_host_rate_limiters = PerHostRateLimiters(
        requests_per_second=PER_HOST_REQUESTS_PER_SECOND,
        jitter_seconds=REQUEST_JITTER_SECONDS,
    )

    url_queue = asyncio.Queue(maxsize=MAX_IN_FLIGHT * 2)
    write_queue = asyncio.Queue(maxsize=WRITE_QUEUE_SIZE)

    start = time.time()

    async with aiohttp.ClientSession(
        connector=connector,
        timeout=timeout,
        raise_for_status=False,
    ) as session:
        with tqdm(total=len(urls)) as pbar:
            fetch_tasks = [
                asyncio.create_task(
                    fetch_worker(
                        url_queue=url_queue,
                        write_queue=write_queue,
                        session=session,
                        global_rate_limiter=global_rate_limiter,
                        per_host_rate_limiters=per_host_rate_limiters,
                    )
                )
                for _ in range(MAX_IN_FLIGHT)
            ]

            write_tasks = [
                asyncio.create_task(
                    write_worker(
                        write_queue=write_queue,
                        results=results,
                        exception_counts=exception_counts,
                        http_status_counts=http_status_counts,
                        user_agent_counts=user_agent_counts,
                        pbar=pbar,
                    )
                )
                for _ in range(WRITE_WORKERS)
            ]

            for image_id, url in enumerate(urls):
                await url_queue.put((image_id, url))

            for _ in fetch_tasks:
                await url_queue.put(None)

            await url_queue.join()
            await asyncio.gather(*fetch_tasks)

            for _ in write_tasks:
                await write_queue.put(None)

            await write_queue.join()
            await asyncio.gather(*write_tasks)

    elapsed = time.time() - start

    write_results_metadata(
        results=results,
        exception_counts=exception_counts,
        http_status_counts=http_status_counts,
        user_agent_counts=user_agent_counts,
        elapsed=elapsed,
    )

    archive_path = make_tar_xz(
        source_dir=SAVE_DIR,
        archive_path=ARCHIVE_PATH,
    )

    ok_count = sum(1 for row in results if row["status"] == "ok")
    error_count = sum(1 for row in results if row["status"] == "error")

    print(f"Downloaded: {ok_count}")
    print(f"Errors: {error_count}")
    print(f"Elapsed: {elapsed:.2f}s")
    print(f"Archive written to: {archive_path}")

    return (
        results,
        exception_counts,
        http_status_counts,
        user_agent_counts,
        elapsed,
        str(archive_path),
    )


# Example call:


In [ ]:

urls = (
    all_data
    .select("image_url")
    .drop_nulls()
    .unique()
    .get_column("image_url")
    .to_list()
)[220_000:330_000]
results, exception_counts, http_status_counts, user_agent_counts, elapsed, archive_path = await fetch_all_images(urls)

In [ ]:
total = len(results)
errors = sum(exception_counts.values())
successes = total - errors

print(f"Images attempted: {total}")

if total > 0:
    print(f"Successful images: {successes} ({successes / total:.2%})")
    print(f"Errors: {errors} ({errors / total:.2%})")
    print(f"Elapsed: {elapsed:.1f}s")
    print(f"Images/sec: {total / elapsed:.2f}")

print("\nErrors by exception type:")
for exception_type, count in exception_counts.most_common():
    print(f"{exception_type}: {count}")

print("\nHTTP errors by status code:")
for status, count in http_status_counts.most_common():
    print(f"{status}: {count}")